# 🧠 Feature Maps in CNNs: Spatial Activation Channels

Welcome to the hands-on explanation notebook for **Feature Maps in CNNs**! In this notebook, we will:
1. Explain the tensor dimensions $[C, H, W]$ of feature maps and how different channels extract distinct visual characteristics.
2. Generate a synthetic image containing shapes (circle and square) using NumPy.
3. Define Sobel horizontal, Sobel vertical, and sharpening filter kernels.
4. Convolve the image with these filters to produce **three distinct feature map channels**.
5. Construct a PyTorch `nn.Conv2d` layer, load our custom filters into its weight tensors, and extract multi-channel feature maps automatically.
6. Plot the feature maps to visualize how CNN layers see edges and textures.
7. Connect feature maps to YOLO's multi-scale prediction layers (P3, P4, P5).

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Set seed for reproducibility
np.random.seed(42)

## 1. Generating a Synthetic Shape Image

We draw a square and a diagonal line on a $64 \times 64$ black canvas to provide features for our filters to extract.

In [ ]:
img = np.zeros((64, 64))

# Draw a square
img[15:45, 15:45] = 1.0

# Draw a diagonal line
for i in range(10, 55):
    img[i, 54 - i] = 1.0

plt.figure(figsize=(5, 5))
plt.imshow(img, cmap='gray')
plt.title('Original Image')
plt.axis('off')
plt.show()

## 2. Defining Custom Kernels (Filters)

We define three kernels:
1.  **Horizontal Edge Detector (Sobel Y):** Detects horizontal boundaries.
2.  **Vertical Edge Detector (Sobel X):** Detects vertical boundaries.
3.  **Sharpen Filter:** Enhances overall contrast.

In [ ]:
k_horizontal = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
])

k_vertical = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
])

k_sharpen = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
])

## 3. Loading Kernels into PyTorch Conv2d Layer

We define a PyTorch Conv2d layer with:
- `in_channels = 1`
- `out_channels = 3` (3 feature map channels)
- `kernel_size = 3`

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=3, kernel_size=3, padding=1, bias=False)

weights = np.stack([k_horizontal, k_vertical, k_sharpen])
weights = weights[:, np.newaxis, :, :]

with torch.no_grad():
    conv.weight.copy_(torch.tensor(weights, dtype=torch.float32))

img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

with torch.no_grad():
    feature_maps = conv(img_tensor).squeeze(0)

print("Output Feature Maps Tensor Shape:", feature_maps.shape)

## 4. Visualizing Feature Map Channels

Let's plot the 3 feature map channels side-by-side.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

titles = [
    "Channel 0: Horizontal Edge Map",
    "Channel 1: Vertical Edge Map",
    "Channel 2: Sharpened Feature Map"
]

for idx in range(3):
    f_map = feature_maps[idx].numpy()
    axes[idx].imshow(f_map, cmap='gray')
    axes[idx].set_title(titles[idx])
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

Look at the channels!
-   **Channel 0:** The top and bottom horizontal lines of the square glow. The vertical lines disappear.
-   **Channel 1:** The left and right vertical edges of the square are highlighted. The horizontal lines disappear.
-   **Channel 2:** Sharpens the entire square, enhancing boundaries.

## 💡 Connection to YOLO and Deep Learning
*   **The Multi-Scale Feature Pyramid:** YOLO extracts multi-scale feature maps at different depths in the network:
    -   **P3 (Stride 8):** High spatial resolution (e.g. $80 \times 80$). Retains detailed edge/coordinate maps, used to detect small objects (like `small-valves`).
    -   **P4 (Stride 16):** Medium resolution ($40 \times 40$).
    -   **P5 (Stride 32):** Low spatial resolution ($20 \times 20$) but high semantic channels (e.g. 512 channels representing complex object parts). Used to classify large objects.
*   By maintaining a hierarchy of feature maps, YOLO can detect objects of varying sizes in a single forward pass.